# Tu copia personal · Aula UTP

Este cuaderno se abre desde GitHub. En Colab elige **Archivo → Guardar una copia en Drive** antes de conservar cambios. Renombra esa copia y ejecuta de arriba abajo. Cada estudiante trabaja en su propio archivo y entorno.

Los ejemplos son didácticos: sigue la sección indicada por Cristian para esta clase. Las comprobaciones son voluntarias, sin entregas ni notas.

# Laboratorio complementario explicado · Machine Learning UTP
Cristian Alvarado. Datos simulados incluidos. Revisión guiada, sin ejercicios para entregar.

Recorrido: preparación → evaluación → ensambles → agrupación por densidad → pronóstico temporal. Ejecuta las celdas en orden.

## Ruta de esta práctica · ML · Encuentros 1–4
1: preparación/regresión; 2: conteos y métricas; 3: ensambles; 4: DBSCAN; 5: series temporales. Los ejemplos de refuerzo y PCA están en cuadernos propios.

**Antes de comenzar:** guarda tu copia personal. Usa el índice lateral de Colab para localizar el bloque indicado. Ejecuta las celdas de preparación y después el bloque, en orden; cambiar una variable exige volver a ejecutar su celda y las que dependen de ella. Abrir una figura no vuelve a entrenar un modelo.

**Entorno:** estos ejemplos están preparados para CPU. No requieren claves de API, contratar servicios ni activar un entorno de pago. La disponibilidad y límites de los recursos de Colab dependen de tu cuenta; no cambies a recursos de pago para esta práctica. Si se interrumpe la sesión, conserva el cuaderno y vuelve a ejecutar desde la preparación.

**Distingue siempre:** datos observados o simulados → parámetros aprendidos por `fit`/entrenamiento → configuraciones que tú eliges → resultados en datos reservados. Una buena cifra de entrenamiento no demuestra aprendizaje útil en nuevos casos.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
from sklearn.cluster import DBSCAN
rng = np.random.default_rng(42)


## 1. ¿Podemos estimar el rendimiento de una finca?

**Pregunta:** con lluvia, temperatura y fertilizante, ¿qué rendimiento estimaría un modelo para una finca que no utilizó al aprender?

Cada fila representa una finca en una temporada. Las entradas **X** son lluvia en **mm**, temperatura en **°C** y fertilizante en **kg/ha**. La respuesta **y** es rendimiento en **t/ha**: toneladas por hectárea.

Usamos **240 fincas simuladas**, no mediciones de campo. Las medidas corresponden a un periodo definido antes de la cosecha y el rendimiento al final de la temporada. Esta simplificación sirve para recorrer el proceso; no es una recomendación de manejo agrícola.

Ejecuta primero la celda de importaciones de arriba. `rng` es el generador de números aleatorios; la semilla 42 permite reproducir el ejemplo al ejecutar el cuaderno desde el comienzo.

### 1.1. Crear la tabla y su respuesta conocida

Generamos lluvia entre 200 y 800 mm, temperatura entre 16 y 30 °C y fertilizante entre 50 y 180 kg/ha. Son rangos elegidos para la simulación, no límites agronómicos generales.

El rendimiento **también se inventa** con una regla: aumenta con lluvia y fertilizante; disminuye al alejarse de 23 °C; lleva una pequeña variación aleatoria. Los números de esa regla son decisiones didácticas. Así conocemos una respuesta para cada fila y podemos comprobar después las predicciones.

`rng.uniform` genera las entradas dentro de los rangos. `rng.normal` añade variación. `**2` eleva al cuadrado la diferencia de temperatura: crea una relación curva. El modelo de regresión aprenderá de las filas; no recibe esta regla generadora.

**Lee la salida:** cada fila muestra las tres entradas y el rendimiento simulado correspondiente.

In [ ]:
n = 240
X = pd.DataFrame({
    'lluvia_mm': rng.uniform(200, 800, n),
    'temperatura_c': rng.uniform(16, 30, n),
    'fertilizante_kg_ha': rng.uniform(50, 180, n)
})
y = (1 + .004 * X.lluvia_mm + .015 * X.fertilizante_kg_ha
     - .03 * (X.temperatura_c - 23)**2 + rng.normal(0, .25, n))
display(X.assign(rendimiento_t_ha=y).head())


### 1.2. Simular registros de lluvia que faltan

Ocultamos la lluvia de **20 fincas**, elegidas al azar sin repetir. `NaN` significa dato ausente; no significa cero lluvia. Mantenemos su rendimiento conocido: pudo registrarse la cosecha aunque faltara la medición de lluvia.

La respuesta se creó **antes** de ocultar los datos para poder evaluar este caso didáctico. Más adelante completaremos las entradas ausentes con una estimación llamada **imputación**.

**Lee la salida:** debe haber 20 valores de lluvia sin registro. Las filas mostradas ayudan a reconocer cómo aparece un dato ausente en una tabla.

In [ ]:
X.loc[rng.choice(n, 20, replace=False), 'lluvia_mm'] = np.nan
print('Fincas sin registro de lluvia:', X.lluvia_mm.isna().sum())
display(X.loc[X.lluvia_mm.isna()].head(3))


### 1.3. Separar antes de aprender

Reservamos **180 fincas para entrenamiento** y **60 para prueba**. `test_size=.25` aparta el 25 %; `random_state=42` reproduce esa partición.

`X_tr` y `y_tr` son entradas y respuestas de entrenamiento. `X_te` y `y_te` son las de prueba. El modelo aprende con las primeras; sus predicciones se comprueban con las respuestas reservadas.

La mediana para completar vacíos y la escala también se aprenden solo con entrenamiento. Usar prueba para calcularlas introduciría información que debía permanecer reservada: **fuga de información**.

Aquí tratamos las fincas como independientes. Para mediciones repetidas de una finca o pronósticos por fecha, habría que separar por finca o respetar el tiempo. En este ejemplo no elegimos hiperparámetros; si comparáramos muchas configuraciones, necesitaríamos validación dentro de entrenamiento.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=.25, random_state=42
)
print('Fincas de entrenamiento:', len(X_tr))
print('Fincas de prueba:', len(X_te))


### 1.4. Crear una referencia sencilla

Antes de usar una regresión, necesitamos una respuesta básica contra la cual comparar. `DummyRegressor(strategy='mean')` aprende la **media del rendimiento de entrenamiento** y predice ese mismo valor para todas las fincas de prueba. No utiliza lluvia, temperatura ni fertilizante para distinguirlas.

`make_pipeline` encadena pasos. En esta referencia, `SimpleImputer(strategy='median')` completa entradas ausentes y después actúa el predictor de media. La imputación mantiene una preparación compatible, aunque este predictor no depende de las entradas.

`fit` aprende esos pasos únicamente con entrenamiento. **Lee la salida:** el número es la predicción de referencia, en t/ha; no es su error.

In [ ]:
base = make_pipeline(
    SimpleImputer(strategy='median'),
    DummyRegressor(strategy='mean')
)
base.fit(X_tr, y_tr)
print(f'La referencia predice {base.predict(X_te)[0]:.3f} t/ha para cada finca.')


### 1.5. Aprender una regresión con las tres entradas

Este segundo pipeline hace tres pasos en orden:

1. **Imputar:** completa una lluvia ausente con la mediana aprendida de las lluvias de entrenamiento.
2. **Estandarizar:** expresa cada entrada respecto a su media y su dispersión, aprendidas con entrenamiento. Así se manejan magnitudes diferentes; esto no añade información ni garantiza una mejora.
3. **Regresión lineal:** aprende un valor inicial y un peso para cada entrada para estimar rendimiento.

Para datos de prueba, el pipeline aplica las mismas reglas y los pesos ya aprendidos; no los recalcula. La relación de temperatura que inventamos es curva y esta regresión usa las tres variables sin potencias: puede quedar error por esa simplificación y por la variación aleatoria.

**Lee la salida:** la mediana está en mm y es la que usará para cualquier lluvia ausente. No tiene por qué ser 600 mm: ese número pertenecía al ejemplo ilustrativo de cuatro fincas de la presentación.

In [ ]:
lineal = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LinearRegression()
)
lineal.fit(X_tr, y_tr)
print(f'Mediana de lluvia aprendida: {lineal[0].statistics_[0]:.3f} mm')


### 1.6. Comparar predicciones con respuestas reservadas

`predict(X_te)` estima rendimientos para las 60 fincas de prueba. Después comparamos cada predicción con su rendimiento conocido, `y_te`. Llamamos **error absoluto** a la distancia entre ambos valores, sin signo; el **MAE** es el promedio de esas distancias: `MAE = promedio(|y − ŷ|)`, donde `ŷ` es la predicción.

**Cómo leer la tabla y el gráfico:**

- Ambos modelos se evalúan sobre las mismas 60 fincas. Cada barra muestra su MAE, no el rendimiento promedio.
- El eje vertical usa t/ha. Un MAE de 0,46 significa un error absoluto promedio cercano a 0,46 toneladas por hectárea en esta prueba; no que todas las fincas tengan ese error.
- La barra más baja corresponde al menor error medio. Compara la regresión con la referencia para saber si las entradas ayudaron en este ejemplo.
- La última tabla muestra cinco fincas de prueba: respuesta conocida, predicción de regresión y error absoluto de cada una.

**Alcance:** con una ejecución completa desde la semilla 42, la referencia obtiene aproximadamente **0,705 t/ha** y la regresión **0,459 t/ha**. La regresión mejora esta referencia en esta simulación. No demuestra que funcionará igual en fincas reales, ni que una variable cause cambios en el rendimiento. Las cifras pueden cambiar si alteras los datos o ejecutas celdas aleatorias de nuevo sin reiniciar el recorrido.

In [ ]:
pred_base = base.predict(X_te)
pred_lineal = lineal.predict(X_te)
resultados = pd.DataFrame({
    'modelo': ['Media de entrenamiento', 'Regresión lineal'],
    'MAE_t_ha': [mean_absolute_error(y_te, pred_base),
                 mean_absolute_error(y_te, pred_lineal)]
})
display(resultados)
plt.bar(resultados.modelo, resultados.MAE_t_ha)
plt.ylabel('MAE en prueba (t/ha)')
plt.title('Comparar con una referencia: mismas 60 fincas de prueba')
plt.show()
comparacion = pd.DataFrame({
    'rendimiento_conocido_t_ha': y_te,
    'prediccion_lineal_t_ha': pred_lineal,
    'error_absoluto_t_ha': np.abs(y_te - pred_lineal)
})
display(comparacion.head().round(3))


### Lectura del gráfico 1 · ¿Las entradas mejoraron la referencia?

**Eje horizontal:** los dos modelos. **Eje vertical:** error absoluto medio en las 60 fincas de prueba, expresado en t/ha. La altura de una barra representa error, no producción ni número de fincas. Cada barra resume 60 comparaciones entre respuesta conocida y predicción.

En esta ejecución, la referencia tiene **0,705 t/ha** de MAE y la regresión **0,459 t/ha**. La segunda barra es más baja: la regresión comete menos error medio sobre la misma prueba. Eso es lo que podemos concluir para este ejemplo.

En la última tabla, una finca tiene rendimiento conocido **4,338**, predicción **4,889** y error absoluto **0,551 t/ha**. La predicción quedó por encima de la respuesta; otra fila puede quedar por debajo. El MAE promedia las distancias sin compensar unos errores con otros. No es un porcentaje de aciertos ni el error de todas las fincas.

**Conexión con la presentación:** la diapositiva 6 resume aprender y predecir; aquí ves las filas, los pasos y el error que produce ese recorrido. La diapositiva 8 usa cuatro fincas ilustrativas para explicar la mediana; este laboratorio calcula su propia mediana con sus 180 fincas de entrenamiento.

Documentación de consulta: [imputación](https://scikit-learn.org/stable/modules/impute.html), [pipelines](https://scikit-learn.org/stable/modules/compose.html#pipeline), [MAE](https://scikit-learn.org/stable/modules/model_evaluation.html#mean-absolute-error).

## 2. Contar errores de clasificación
En este ejemplo de 10 plantas, positivo significa con plaga. Filas = realidad, columnas = predicción; el orden es sin plaga, con plaga. Cuenta TP=3, FN=1, FP=2 y TN=4. Mira por qué precision y recall responden preguntas distintas.

In [ ]:
real=np.array([1,1,1,1,0,0,0,0,0,0])
pred=np.array([1,1,1,0,1,1,0,0,0,0])
cm=confusion_matrix(real,pred,labels=[0,1])
display(pd.DataFrame(cm,index=['Real sin plaga','Real con plaga'],columns=['Pred sin plaga','Pred con plaga']))
display(pd.DataFrame({'métrica':['Accuracy','Precision','Recall','F1'],'valor':[accuracy_score(real,pred),precision_score(real,pred),recall_score(real,pred),f1_score(real,pred)]}))
fig,ax=plt.subplots();ax.imshow(cm,cmap='Blues')
for i in range(2):
    for j in range(2): ax.text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=20)
ax.set_xticks([0,1],['Sin plaga','Con plaga']);ax.set_yticks([0,1],['Sin plaga','Con plaga']);ax.set_xlabel('Predicción');ax.set_ylabel('Realidad');ax.set_title('Matriz de confusión · 10 plantas');plt.show()


### Lectura del gráfico 2 · ¿Qué errores cometimos en diez plantas?

Esta imagen es una **matriz de conteos**, no una curva. El eje vertical indica la **realidad** y el horizontal la **predicción**. Para leer una casilla, combina su fila y su columna. El azul más oscuro significa más plantas en esa casilla, no mayor calidad.

| Realidad → predicción | Plantas | Interpretación |
|---|---:|---|
| Sin plaga → sin plaga | 4 | Sanas correctamente descartadas (TN). |
| Sin plaga → con plaga | 2 | Falsas alertas (FP). |
| Con plaga → sin plaga | 1 | Plaga que se pasó por alto (FN). |
| Con plaga → con plaga | 3 | Plagas detectadas (TP). |

La diagonal reúne los **7 aciertos**. Fuera de ella están **3 errores**, con consecuencias distintas: revisar una planta sana o dejar pasar una planta con plaga.

La tabla de métricas usa decimales: **accuracy 0,70 = 70 %** de aciertos; **precision 0,60 = 60 %** de las alertas correctas; **recall 0,75 = 75 %** de las plantas con plaga detectadas. **F1 ≈ 0,667** es su media armónica: `F1 = 2 × precision × recall / (precision + recall)`. Ninguna métrica por sí sola dice cuánto cuesta cada error. Estas diez respuestas son un ejemplo construido; aquí no entrenamos un clasificador.

## 3. Comparar ensambles sobre la misma prueba
Reutilizamos las fincas y la partición del bloque 1. Árbol = modelo individual; Random Forest combina árboles con aleatoriedad; gradient boosting los construye en secuencia. Revisa MAE y RMSE frente a la referencia. El orden de las barras es el de esta ejecución, no una superioridad universal.

In [ ]:
modelos={'Referencia':base,'Árbol':make_pipeline(SimpleImputer(strategy='median'),DecisionTreeRegressor(max_depth=5,random_state=42)),'Random Forest':make_pipeline(SimpleImputer(strategy='median'),RandomForestRegressor(n_estimators=100,max_depth=6,random_state=42)),'Gradient boosting':make_pipeline(SimpleImputer(strategy='median'),GradientBoostingRegressor(random_state=42))}
filas=[]
for nombre,modelo in modelos.items():
    modelo.fit(X_tr,y_tr);p=modelo.predict(X_te)
    filas.append({'modelo':nombre,'MAE_t_ha':mean_absolute_error(y_te,p),'RMSE_t_ha':np.sqrt(mean_squared_error(y_te,p)),'R2':r2_score(y_te,p)})
tabla=pd.DataFrame(filas);display(tabla)
plt.barh(tabla.modelo,tabla.MAE_t_ha);plt.xlabel('MAE en prueba (t/ha)');plt.title('Mismos datos de prueba para todos');plt.show()


### Lectura del gráfico 3 · Comparar modelos con la misma regla

Las barras son **horizontales**: el eje vertical contiene los nombres y el horizontal el **MAE en prueba, en t/ha**. Una barra más corta significa menor error medio. El orden de arriba abajo no es una escala de calidad: lee el nombre y la longitud de cada barra.

En esta ejecución: referencia **0,705**, árbol **0,437**, Random Forest **0,358** y gradient boosting **0,301 t/ha**. Gradient boosting tiene el menor MAE entre estos cuatro modelos, con estas configuraciones y estas 60 fincas de prueba.

La tabla añade **RMSE**, también en t/ha, que da más peso a errores grandes, y **R²**, que compara el error cuadrático con la variación de las respuestas de prueba. En fórmulas: `RMSE = √promedio((y − ŷ)²)` y `R² = 1 − suma((y − ŷ)²) / suma((y − media_de_prueba)²)`. R² no es un porcentaje de aciertos y puede ser negativo. El gráfico solo representa MAE: sus barras no muestran RMSE ni R².

La referencia usa la media aprendida en entrenamiento. No cambia su regla para aprovechar la prueba. Una sola comparación no demuestra que un algoritmo será mejor en cualquier cultivo, periodo o conjunto de fincas.

## 4. DBSCAN: grupos y puntos no agrupados
Simulamos dos concentraciones de mediciones y algunos puntos aislados. Escalamos antes de agrupar. eps controla cercanía y min_samples exige densidad. −1 significa no agrupado con esta configuración; no significa automáticamente error del sensor.

In [ ]:
a=rng.normal([0,0],.25,(60,2));b=rng.normal([2,2],.25,(60,2));aislados=np.array([[-2,2],[4,-1],[0,4]])
datos=np.vstack([a,b,aislados]);escalados=StandardScaler().fit_transform(datos)
etiquetas=DBSCAN(eps=.35,min_samples=5).fit_predict(escalados)
display(pd.Series(etiquetas).value_counts().sort_index().rename('número de puntos'))
for grupo in np.unique(etiquetas):
    sel=etiquetas==grupo;plt.scatter(datos[sel,0],datos[sel,1],label='No agrupado' if grupo==-1 else f'Grupo {grupo}')
plt.xlabel('Medida 1 (simulada)');plt.ylabel('Medida 2 (simulada)');plt.title('DBSCAN: agrupación por densidad');plt.legend();plt.show()


### Lectura del gráfico 4 · Reconocer grupos sin confundirlos con categorías conocidas

**Cada punto** es una observación con dos medidas inventadas. El eje horizontal es **Medida 1** y el vertical **Medida 2**. No son lluvia ni temperatura y no les asignamos unidades físicas.

La leyenda distingue **Grupo 0**, **Grupo 1** y **No agrupado**. Los números de grupo son identificadores: no significan calidad baja o alta. En esta ejecución hay **60 puntos en el grupo 0**, **57 en el grupo 1** y **6 no agrupados**. La tabla confirma esos conteos; −1 identifica los no agrupados.

DBSCAN se calculó sobre medidas **estandarizadas**, pero el gráfico muestra las coordenadas originales para facilitar la lectura. Por eso `eps=0,35` no se puede interpretar directamente como un radio de 0,35 en los ejes dibujados.

Busca concentraciones y puntos apartados, y confirma el resultado con la leyenda. «No agrupado» significa que un punto no pertenece a un grupo con esta configuración; no demuestra que el sensor esté mal ni que el punto deba eliminarse. Cambiar la escala, eps o min_samples puede cambiar los grupos.

## 5. Series temporales: comprobar el futuro
Simulamos 84 semanas de ventas. Las primeras 60 entrenan; las últimas 24 evalúan. Comparamos una recta con repetir el último valor y con usar la misma semana de la temporada anterior (período 12). Es un pronóstico fijo de 24 semanas. Todas las entradas de las referencias estaban disponibles al cerrar entrenamiento. Observa tendencia y ciclo: una recta no representa la estacionalidad.

In [ ]:
t=np.arange(84)
ventas=100+.6*t+12*np.sin(2*np.pi*t/12)+rng.normal(0,2,len(t))
corte=60;periodo=12
pasado=ventas[:corte];futuro=ventas[corte:]
t_tr=t[:corte,None];t_te=t[corte:,None]
modelo=LinearRegression().fit(t_tr,pasado)
pronosticos={'Último valor':np.repeat(pasado[-1],len(futuro)),'Referencia estacional':np.tile(pasado[-periodo:],2),'Tendencia lineal':modelo.predict(t_te)}
display(pd.DataFrame({'método':list(pronosticos),'MAE_unidades':[mean_absolute_error(futuro,p) for p in pronosticos.values()]}))
plt.figure(figsize=(10,4));plt.plot(t,ventas,label='Ventas observadas',color='black')
for nombre,p in pronosticos.items(): plt.plot(t[corte:],p,label=nombre)
plt.axvline(corte-.5,linestyle='--',color='gray',label='Fin de entrenamiento');plt.xlabel('Semana');plt.ylabel('Unidades vendidas');plt.title('Pasado para entrenar, futuro para evaluar');plt.legend();plt.show()


### Lectura del gráfico 5 · Comparar pronósticos con lo que después ocurrió

**Eje horizontal:** semana, con índices 0 a 83. **Eje vertical:** unidades vendidas simuladas. La línea negra muestra las ventas observadas. La línea gris discontinua separa las **60 semanas conocidas al entrenar** de las **24 semanas reservadas para evaluar**.

Las tres líneas de pronóstico comienzan después de ese corte:

- **Último valor:** repite la última venta conocida; por eso es horizontal.
- **Referencia estacional:** repite el último ciclo de 12 semanas conocido; representa el ciclo sin añadir crecimiento futuro.
- **Tendencia lineal:** prolonga una recta aprendida con el pasado; representa crecimiento, pero no las oscilaciones del ciclo.

Usa la **leyenda** para identificar las líneas. A la derecha del corte, compara cada pronóstico con la línea negra. La separación vertical es la diferencia en unidades vendidas de esa semana. Las ventas negras de ese tramo se muestran para evaluar; no se usaron al generar los pronósticos.

La tabla resume las 24 diferencias: en esta ejecución, MAE aproximado **17,28** para último valor, **11,34** para referencia estacional y **7,51 unidades** para tendencia lineal. La recta tiene menor error medio en esta simulación, aunque no dibuje el ciclo. Un parecido visual parcial no sustituye medir todos los errores.

Son **24 pronósticos hechos desde un único corte**, no predicciones que se actualizan cada semana con nuevas ventas observadas. No son datos comerciales reales ni una garantía para otra serie.

## Cierre de comprensión
Relaciona cada resultado con su pregunta: ¿qué se predice?, ¿qué datos aprendieron la preparación?, ¿qué significa cada eje?, ¿qué referencia se comparó?, ¿qué error importa? La conversación se realiza con el docente, sin entregables.

Fuentes: [scikit-learn: errores frecuentes](https://scikit-learn.org/stable/common_pitfalls.html), [evaluación](https://scikit-learn.org/stable/modules/model_evaluation.html), [validación temporal](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.TimeSeriesSplit.html).